# Introduction
As we have completed a preprocessing pipeline, it is now time to select a model, perform feature 
tuning, and perform hyperparameter tunining. We will measure models on the following metrics:
- Precision (focused, it is more crucial to be correct than to flag all spam)
- Recall
- ROC Curve
- Accuracy

## Setup

In [40]:
# use this cell to install all requirements for this project.
# !pip install -r requirements.txt

In [41]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [42]:
# make array print readable
np.set_printoptions(suppress=True, precision=4)

In [43]:
df = pd.read_csv(
    'data/cleaned_emails.csv', 
    index_col='filename', )
df.head()

,body,from,weekday,day,month,year,hour,timezone,to,subject,return_path,received,delivered_to,message_id,spam
filename,,,,,,,,,,,,,,,
01128.efb36914ecb55d78a894591eff0843c5,"['on', 'sun', 'NUMBER', 'jul', 'NUMBER', 'NUMB...",uni.de,sun,21,jul,2002,20,-400.0,freshrpms.net,"['re', 'ximian', 'apt', 'repo']",freshrpms.net,7,1,uni.de,False
00659.02e6dd777f837798533eae8f3b6a0491,"['what', 'is', 'mime', 'mime', 'stand', 'for',...",docserver.cac.washington.edu,mon,19,aug,2002,23,-700.0,example.sourceforge.net,"['wm', 'the', 'mime', 'inform', 'you', 'reques...",example.sourceforge.net,6,1,docserver.cac.washington.edu,False
00776.7df92458e9cf04b8873c406bde7d2fbe,"['im', 'not', 'up', 'to', 'fork', 'the', 'text...",golux.com,tue,13,aug,2002,15,-400.0,xent.com,"['a', 'messag', 'for', 'our', 'time']",xent.com,6,2,golux.com,False
00116.409b29c26edef06268b4bfa03ef1367a,"['on', 'sat', 'jul', 'NUMBER', 'NUMBER', 'at',...",skynet.ie,sat,20,jul,2002,13,100.0,linux.ie,"['re', 'ilug', 'vanquish', 'the', 'daemon', 'o...",linux.ie,8,1,skynet.ie,False
00615.23556d88fcb1179b25083cfc41017f42,"['origin', 'messag', 'date', 'thu', 'NUMBER', ...",dmv.com,thu,8,aug,2002,16,-400.0,example.sourceforge.net,"['re', 'razorus', 'use', 'razor', 'with', 'non...",example.sourceforge.net,7,1,landshark,False


In [44]:
# import feature pipeline
import cloudpickle as cp

feature_pipeline = None
with open('objects/feature_pipeline.pkl', 'rb') as f:
    feature_pipeline = cp.load(f)

In [45]:
# creating train-test split
from sklearn.model_selection import train_test_split

X = df.drop('spam', axis=1)
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

print(X_train.head(1))
print()
print(y_train.head(1))

                                                                                     body  \
filename                                                                                    
00876.668b933f0dcc4bbbc7bad7255ba2d659  ['unless', 'your', 'parent', 'were', 'select',...   

                                              from weekday  day month  year  \
filename                                                                      
00876.668b933f0dcc4bbbc7bad7255ba2d659  silcom.com     wed   24   jul  2002   

                                        hour  timezone  \
filename                                                 
00876.668b933f0dcc4bbbc7bad7255ba2d659    17    -700.0   

                                                            to  \
filename                                                         
00876.668b933f0dcc4bbbc7bad7255ba2d659  spamassassin.taint.org   

                                                                                  subject  \
filename       

# Model Selection
Here, we will perform cross-validation on a variety of different models using standard parameters.
Given our current data set of scaled variables, linear models seem promising to creating a great
classifier. Models like Decision Trees, however, may struggle due to our number of features and due
to the use of cyclical encoding. We will test our data against the following models:
- Logistic Regression
- Stochastic Gradient Descent
- Support Vector Machines
- RandomForestClassifiers

In [73]:
from sklearn.model_selection import cross_val_predict, cross_val_score

def test_model(pipe, cv=5, scoring='f1'):
    
    print('~ CROSS VALIDATION SCORES ~\n')
    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        verbose=3,
    )

    print(cv_scores)
    print(f'mean {scoring} score: {np.mean(cv_scores)}')

feature_pipeline.verbose = False

## Logistic Regression

In [74]:
# create logistic regression pipeline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

log_clf_pipeline = Pipeline([
    ('trans', feature_pipeline),
    ('log_clf', LogisticRegression())
])


test_model(log_clf_pipeline)

~ CROSS VALIDATION SCORES ~

[Pipeline] ........ (step 1 of 3) Processing num_impute, total=   0.0s
[Pipeline]  (step 2 of 3) Processing create_path_length, total=   0.0s
[Pipeline] ......... (step 3 of 3) Processing num_scale, total=   0.0s
[Pipeline] ........ (step 1 of 3) Processing cat_impute, total=   0.0s
[Pipeline] ....... (step 2 of 3) Processing cat_ordinal, total=   0.0s
[Pipeline] ...... (step 3 of 3) Processing cat_cyclical, total=   0.0s
[Pipeline] ........ (step 1 of 2) Processing day_impute, total=   0.0s
[Pipeline] ...... (step 2 of 2) Processing day_cyclical, total=   0.0s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.7s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.1s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.0s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.0s
[CV] END ................................ score: (test=0.955) total time=   1.2s
[Pipeline] ........ (step 1 of 3) Proc

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    5.8s finished


## SGDClassifier

In [68]:
from sklearn.linear_model import SGDClassifier

sgd_clf_pipeline = Pipeline([
    ('trans', feature_pipeline),
    ('sgd_clf', SGDClassifier())
])

test_model(sgd_clf_pipeline)

~ CROSS VALIDATION SCORES ~

[Pipeline] ........ (step 1 of 3) Processing num_impute, total=   0.0s
[Pipeline]  (step 2 of 3) Processing create_path_length, total=   0.0s
[Pipeline] ......... (step 3 of 3) Processing num_scale, total=   0.0s
[Pipeline] ........ (step 1 of 3) Processing cat_impute, total=   0.0s
[Pipeline] ....... (step 2 of 3) Processing cat_ordinal, total=   0.0s
[Pipeline] ...... (step 3 of 3) Processing cat_cyclical, total=   0.0s
[Pipeline] ........ (step 1 of 2) Processing day_impute, total=   0.0s
[Pipeline] ...... (step 2 of 2) Processing day_cyclical, total=   0.0s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.8s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.1s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.0s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.0s
[CV] END ................................ score: (test=0.955) total time=   1.2s
[Pipeline] ........ (step 1 of 3) Proc

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    6.0s finished


## Support Vector Machine

In [69]:
from sklearn.svm import SVC

svc_clf_pipeline = Pipeline([
    ('trans', feature_pipeline),
    ('svm_clf', SVC())
])

test_model(svc_clf_pipeline)

~ CROSS VALIDATION SCORES ~

[Pipeline] ........ (step 1 of 3) Processing num_impute, total=   0.0s
[Pipeline]  (step 2 of 3) Processing create_path_length, total=   0.0s
[Pipeline] ......... (step 3 of 3) Processing num_scale, total=   0.0s
[Pipeline] ........ (step 1 of 3) Processing cat_impute, total=   0.0s
[Pipeline] ....... (step 2 of 3) Processing cat_ordinal, total=   0.0s
[Pipeline] ...... (step 3 of 3) Processing cat_cyclical, total=   0.0s
[Pipeline] ........ (step 1 of 2) Processing day_impute, total=   0.0s
[Pipeline] ...... (step 2 of 2) Processing day_cyclical, total=   0.0s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.7s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.2s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.0s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.0s
[CV] END ................................ score: (test=0.957) total time=   1.2s
[Pipeline] ........ (step 1 of 3) Proc

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    6.2s finished


## Random Forest Classifier

In [70]:
from sklearn.ensemble import RandomForestClassifier

rf_clf_pipeline = Pipeline([
    ('trans', feature_pipeline),
    ('rf_clf', RandomForestClassifier())
])

test_model(rf_clf_pipeline)

~ CROSS VALIDATION SCORES ~

[Pipeline] ........ (step 1 of 3) Processing num_impute, total=   0.0s
[Pipeline]  (step 2 of 3) Processing create_path_length, total=   0.0s
[Pipeline] ......... (step 3 of 3) Processing num_scale, total=   0.0s
[Pipeline] ........ (step 1 of 3) Processing cat_impute, total=   0.0s
[Pipeline] ....... (step 2 of 3) Processing cat_ordinal, total=   0.0s
[Pipeline] ...... (step 3 of 3) Processing cat_cyclical, total=   0.0s
[Pipeline] ........ (step 1 of 2) Processing day_impute, total=   0.0s
[Pipeline] ...... (step 2 of 2) Processing day_cyclical, total=   0.0s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.7s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.1s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   0.0s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.0s
[CV] END ................................ score: (test=0.965) total time=   1.4s
[Pipeline] ........ (step 1 of 3) Proc

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    7.4s finished


## Conclusions
Looking at our initial model tests, we have the following data:

| Model | F1 Mean |
|---|---|
| Logistic Regression | 0.9453 |
| SGD Classifier | 0.9432 |
| SVM | 0.9528 |
| Random Forest | 0.9612 |

With these results, we can see our highest performing models. Focusing on the F1-score, we result
with the Random Forest Classifier and SVM at very high scores with our training data. These models, 
off initial training already perform exceptionally, yet a greater amount of fine tuning is beneficial
and will be performed.